In [ ]:
from langgraph.graph import MessagesState,START,StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage,RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
load_dotenv()

In [ ]:
model=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
class ChatState(MessagesState):
    summary:str

In [ ]:
def chat_node(state:ChatState):
    messages=[]
    if state["summary"]:
        messages.append({
            "role":"system",
            "content":f"Conversation summary:\n {state['summary']}"
        })
    messages.extend(state["messages"])
    print(messages)
    response=model.invoke(messages)
    return {"messages":[response]}

In [ ]:
def summarize_conversation(state:ChatState):
    existing_summary=state["summary"]
    if existing_summary:
        prompt={
            f"Existing summary: {existing_summary}\n\n"
            "extend the summary using new conversation above"
        }
    else:
        prompt="Summarize the conversation above"

    messages_for_summary=state["messages"]+[HumanMessage(content=prompt)]
    response=model.invoke(messages_for_summary)

    messages_to_delete=state["messages"][:-2]
    return {
        "summary":response.content,
        "messages":[RemoveMessage(id=m.id)for m in messages_to_delete]
    }

In [ ]:
builder=StateGraph(ChatState)
builder.add_node("chat",chat_node)
builder.add_edgr("summarize",summarize_conversation)
builder.add_edge(START,"chat")
builder.add_conditional_edges("chat",should_summarize,{True:"summarize",False:"__end__"})
builder.add_edge("summarize","__end__")

In [ ]:
checkpointer=InMemorySaver()    
graph=builder.compile(checkpointer=checkpointer)

In [ ]:
config={"configurable":{"thread_id":"t1"}}
